Notebook to generate data.h5 that includes curve_data, sample_info and igi_gene_call datasets

In [59]:
import os
import json
import glob
import pandas as pd
import numpy as np

In [39]:
DATA_DIR = '/Users/huongvu/Desktop/pcr/Data/New Data/'
CURVE_DIR = os.path.join(DATA_DIR, 'extracted_csv')
PRIMARY_DIR = os.path.join(DATA_DIR, 'primary_samples')
SIM_DIR = '/Users/huongvu/Desktop/PCR_simulation/'


In [40]:
sample_info = pd.read_csv(os.path.join(PRIMARY_DIR, 'sample_info.csv'))
sample_gene = pd.read_csv(os.path.join(PRIMARY_DIR, 'sample_gene.csv'))

In [41]:
sample_gene['rank'] = (sample_gene
                       .groupby(['sample_id','pcr_plate','target'])['thres_ct']
                       .rank(method='dense', ascending=False))
sample_gene = sample_gene.loc[sample_gene['rank'] == 1.0]
sample_gene = sample_gene.drop(labels='rank', axis=1)

In [42]:
sample_info['rank'] = (sample_info
                       .groupby(['sample_id','pcr_plate'])['created_date']
                       .rank(method='dense', ascending=False))
sample_info = sample_info.loc[sample_info['rank'] == 1.0]
sample_info = sample_info.drop(labels='rank', axis=1)

In [43]:
folders = os.listdir(CURVE_DIR)
folders.remove('.DS_Store')

amp_df = pd.DataFrame()
multcomp_df = pd.DataFrame()
da_result_df = pd.DataFrame()

error_files = {'amplification':[],
               'multicomponent':[],
               'da_result':[]}

In [44]:
for folder in folders:
    try: 
        df = pd.read_csv(os.path.join(CURVE_DIR, folder, 'Amplification Data.csv'))
        df = df.loc[:, ~df.columns.isin(['Well','Sample','Omit'])]
        df.columns = ['well_position','cycle_no','target','rn','drn']
        df.loc[:,'file'] = folder
        amp_df = pd.concat([amp_df, df],
                            ignore_index=True)
    except Exception as e:
        error_files['amplification'].append({'folder':folder,
                                          'msg': str(e)}) 
    
    try:
        df = pd.read_csv(os.path.join(CURVE_DIR, folder, 'Multicomponent.csv'))
        df = df.drop(columns='Well')
        df = (df.set_index(['Well Position','Cycle Number'])
                .melt(ignore_index=False)
                .reset_index())
        df.columns = ['well_position','cycle_no','dye','Fn']
        df.loc[:,'file'] = folder
        multcomp_df = pd.concat([multcomp_df, df],
                                ignore_index=True)
    except Exception as e:
        error_files['multicomponent'].append({'folder':folder,
                                          'msg': str(e)})

    try:
        df = pd.read_csv(os.path.join(CURVE_DIR, folder, 'Results.csv'))
        df = df[['Well Position','Target','Reporter','Amp Score','Cq',
                 'Threshold','Baseline Start','Baseline End']]
        df.columns = ['well_position','target','dye','amp_score','cq',
                      'threshold','baseline_start','baseline_end']
        df.loc[:,'file'] = folder
        da_result_df = pd.concat([da_result_df, df],
                                 ignore_index=True)
    except Exception as e:
        error_files['da_result'].append({'folder':folder,
                                          'msg': str(e)})

In [45]:
# check for error files
print(error_files)

{'amplification': [], 'multicomponent': [], 'da_result': []}


In [46]:
curve_df = (da_result_df
           .merge(amp_df, how = 'inner', on = ['well_position','file','target'])
           .merge(multcomp_df, how = 'inner', on = ['well_position','file','dye','cycle_no']))

file_dict = sample_info[['file','pcr_plate']].drop_duplicates().set_index('file').to_dict()['pcr_plate']
curve_df['pcr_plate'] = curve_df['file'].replace(file_dict)

curve_df['curve_idx'] = curve_df.groupby(['pcr_plate','target','well_position']).ngroup()
curve_df = curve_df.drop('file', axis = 1)

sample_info = sample_info.drop(sample_info[sample_info.pcr_plate == 'AC00DB15'].index, axis=0)
curve_df = curve_df.drop(curve_df[curve_df.pcr_plate == 'AC00DB15'].index, axis = 0)

Drop pcr_plate AC00GXWF, AC00DB6F, AC00H0OU because the whole plate is invalid

In [47]:
curve_df = curve_df.drop(curve_df[curve_df.pcr_plate.isin(['AC00GXWF','AC00DB6F','AC00H0OU'])].index, axis=0)
sample_info = sample_info.drop(sample_info[sample_info.pcr_plate.isin(['AC00GXWF','AC00DB6F','AC00H0OU'])].index, axis=0)
sample_gene = sample_gene.drop(sample_gene[sample_gene.pcr_plate.isin(['AC00GXWF','AC00DB6F','AC00H0OU'])].index, axis=0)

Drop sample_ids that are part of a pooled sample

In [48]:
sample_info = sample_info.drop(sample_info[sample_info.sample_id.isin(['S1268904', 'S1268905'])].index, axis=0)

impute current_sample_result for positive and negative control wells

In [50]:
sample_info.loc[sample_info.sample_type == 'Positive Control (qPCR)', 'current_sample_result'] = 'Positive'
sample_info.loc[sample_info.sample_type == 'Negative Control (qPCR)', 'current_sample_result'] = 'Negative'

impute Na retest sample id with np.nan

In [61]:
sample_info.loc[sample_info['retest_sample_id_1'] == ' ', 'retest_sample_id_1'] = np.nan
sample_info.loc[sample_info['retest_sample_id_2'] == ' ', 'retest_sample_id_2'] = np.nan

In [23]:
curve_df.to_hdf(os.path.join(SIM_DIR, 'data', 'data.h5'), key = 'curve_data', mode='w')
sample_info.to_hdf(os.path.join(SIM_DIR, 'data', 'data.h5'), key='sample_info')
sample_gene.to_hdf(os.path.join(SIM_DIR, 'data', 'data.h5'), key='igi_gene_call')

/opt/anaconda3/envs/pcr/lib/python3.8/site-packages/pandas/core/generic.py:2703: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block0_values] [items->Index(['sample_id', 'sample_barcode', 'pcr_plate', 'well_position',
       'sample_type', 'final_patient_result', 'current_sample_result',
       'created_date', 'record_type', 'retest_sample_id_1',
       'retest_sample_id_2', 'file'],
      dtype='object')]

  pytables.to_hdf(


In [63]:
join_df = (curve_df # exclude pooled samples
    .merge(sample_info, how='inner', on=['well_position','pcr_plate'])
    .merge(sample_gene, how='inner', on=['pcr_plate','sample_id','target']))
join_df.head()

,well_position,target,dye,amp_score,cq,threshold,baseline_start,baseline_end,cycle_no,rn,...,sample_type,final_patient_result,current_sample_result,created_date,record_type,retest_sample_id_1,retest_sample_id_2,file,igi_call,thres_ct
0,A1,S gene,ABY,0.0,Undetermined,8810.498,3,39,1,147103.828125,...,Clinical Sample,Negative,Negative,44205,Pooled Sample,NaN,NaN,db1i,Negative,Undetermined
1,A1,S gene,ABY,0.0,Undetermined,8810.498,3,39,2,146864.656250,...,Clinical Sample,Negative,Negative,44205,Pooled Sample,NaN,NaN,db1i,Negative,Undetermined
2,A1,S gene,ABY,0.0,Undetermined,8810.498,3,39,3,146410.109375,...,Clinical Sample,Negative,Negative,44205,Pooled Sample,NaN,NaN,db1i,Negative,Undetermined
3,A1,S gene,ABY,0.0,Undetermined,8810.498,3,39,4,146188.328125,...,Clinical Sample,Negative,Negative,44205,Pooled Sample,NaN,NaN,db1i,Negative,Undetermined
4,A1,S gene,ABY,0.0,Undetermined,8810.498,3,39,5,146078.421875,...,Clinical Sample,Negative,Negative,44205,Pooled Sample,NaN,NaN,db1i,Negative,Undetermined


In [52]:
join_df[(join_df.cycle_no == 1) & (join_df.current_sample_result == 'Invalid')].groupby('sample_type').count()

,well_position,target,dye,amp_score,cq,threshold,baseline_start,baseline_end,cycle_no,rn,...,sample_barcode,final_patient_result,current_sample_result,created_date,record_type,retest_sample_id_1,retest_sample_id_2,file,igi_call,thres_ct
sample_type,,,,,,,,,,,,,,,,,,,,,
Buffer Negative Control (Extraction),28,28,28,28,28,28,28,28,28,28,...,28,12,28,28,28,28,28,28,28,28
Clinical Sample,4118,4118,4118,4118,4118,4118,4118,4118,4118,4118,...,4118,3894,4118,4118,4118,4118,4118,4118,4118,4118
Human Normal Negative Control (Extraction),20,20,20,20,20,20,20,20,20,20,...,20,8,20,20,20,20,20,20,20,20


In [53]:
join_df.groupby(['sample_type', 'current_sample_result']).curve_idx.nunique()

sample_type                                 current_sample_result
Buffer Negative Control (Extraction)        Invalid                     28
                                            Negative                   228
Clinical Sample                             Inconclusive              1859
                                            Invalid                   4118
                                            Negative                 54338
                                            Positive                  5229
Human Normal Negative Control (Extraction)  Invalid                     20
                                            Negative                   236
Negative Control (qPCR)                     Negative                  1504
Positive Control (qPCR)                     Positive                  1504
Name: curve_idx, dtype: int64

In [54]:
join_df[(join_df.sample_type == 'Clinical Sample')].curve_idx.nunique()

65604

In [80]:
join_df[(join_df.current_sample_result == 'Inconclusive') & (join_df.cycle_no == 45) & (join_df.sample_type == 'Clinical Sample')].groupby(['sample_id']).count()

,well_position,target,dye,amp_score,cq,threshold,baseline_start,baseline_end,cycle_no,rn,drn,Fn,pcr_plate,curve_idx,sample_barcode,sample_type,final_patient_result,current_sample_result,created_date,record_type,retest_sample_id_1,retest_sample_id_2,file,igi_call,thres_ct
sample_id,,,,,,,,,,,,,,,,,,,,,,,,,
S1005189,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,0,0,3,3,3
S1005232,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,0,0,3,3,3
S1015907,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,0,3,3,3
S1016033,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,0,3,3,3
S1017975,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,0,3,3,3
S1018129,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,0,3,3,3
S1018703,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,0,0,3,3,3
S1019512,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,0,0,3,3,3
S1019514,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,0,0,3,3,3


In [86]:
df = join_df[(join_df.current_sample_result == 'Inconclusive') & (join_df.sample_type == 'Clinical Sample') & (join_df.cycle_no == 45)].copy()
df['igi_call_encoded'] = [1 if x == 'Positive' else 0 for x in df['igi_call']]
df[df.target != 'RnaseP'].groupby('sample_id').igi_call_encoded.sum().count()
# join_df[(join_df.cycle_no == 45)].groupby('sample_id').agg(vote = ('igi_call', lambda x: sum(1 if x == 'Positive' else 0)))

405

In [84]:
df[df.sample_id.isin(['S993850'])]

,well_position,target,dye,amp_score,cq,threshold,baseline_start,baseline_end,cycle_no,rn,drn,Fn,pcr_plate,curve_idx,sample_id,sample_barcode,sample_type,final_patient_result,current_sample_result,created_date,record_type,retest_sample_id_1,retest_sample_id_2,file,igi_call,thres_ct,igi_call_encoded
1915094,K1,RnaseP,ATTO 647,2.604354,22.303629600783268,10000.0,3,14,45,341940.781250,311271.214714,341940.780,AC00GXZR,63616,S993850,SARS00016081,Clinical Sample,Negative,Inconclusive,12/16/21,Submitted Sample,SARS00016081- Retest 1,SARS00016081- Retest 2,gxzr,Positive,22.3036296,1
1915139,K1,E gene,VIC,0.000000,Undetermined,6000.0,3,44,45,217498.609375,597.350446,217498.610,AC00GXZR,62848,S993850,SARS00016081,Clinical Sample,Negative,Inconclusive,12/16/21,Submitted Sample,SARS00016081- Retest 1,SARS00016081- Retest 2,gxzr,Negative,Undetermined,0
1915184,K1,N gene,FAM,1.843257,33.39960587063479,11000.0,3,15,45,125533.304688,12495.643115,125533.305,AC00GXZR,63232,S993850,SARS00016081,Clinical Sample,Negative,Inconclusive,12/16/21,Submitted Sample,SARS00016081- Retest 1,SARS00016081- Retest 2,gxzr,Positive,33.39960587,1


In [74]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
join_df[(join_df.current_sample_result == 'Inconclusive') & (join_df.cycle_no == 45)]

,well_position,target,dye,amp_score,cq,threshold,baseline_start,baseline_end,cycle_no,rn,drn,Fn,pcr_plate,curve_idx,sample_id,sample_barcode,sample_type,final_patient_result,current_sample_result,created_date,record_type,retest_sample_id_1,retest_sample_id_2,file,igi_call,thres_ct
57504,G15,RnaseP,ATTO 647,2.645879,25.142049385208686,10000.000000,3,17,45,414146.937500,385517.983219,414146.940,AC00GY8Y,71590,S1075558,SARS00035536,Clinical Sample,Negative,Inconclusive,1/21/22,Submitted Sample,SARS00035536- Retest 1,NaN,gy8y,Positive,25.14204939
57549,G15,E gene,VIC,1.875754,Undetermined,6000.000000,3,44,45,332594.531250,4330.283174,332594.530,AC00GY8Y,70822,S1075558,SARS00035536,Clinical Sample,Negative,Inconclusive,1/21/22,Submitted Sample,SARS00035536- Retest 1,NaN,gy8y,Negative,Undetermined
57594,G15,N gene,FAM,2.083967,35.733493614036334,12500.000000,3,28,45,151175.140625,41901.365887,151175.140,AC00GY8Y,71206,S1075558,SARS00035536,Clinical Sample,Negative,Inconclusive,1/21/22,Submitted Sample,SARS00035536- Retest 1,NaN,gy8y,Positive,35.73349361
62364,M11,RnaseP,ATTO 647,2.636645,23.24589169430029,10000.000000,3,16,45,402540.718750,373809.225348,402540.720,AC00GY8Y,71730,S1075521,SARS00032181,Clinical Sample,Positive,Inconclusive,1/21/22,Submitted Sample,SARS00032181- Retest 1,NaN,gy8y,Positive,23.24589169
62409,M11,E gene,VIC,1.873418,Undetermined,6000.000000,3,44,45,308078.062500,2492.906867,308078.060,AC00GY8Y,70962,S1075521,SARS00032181,Clinical Sample,Positive,Inconclusive,1/21/22,Submitted Sample,SARS00032181- Retest 1,NaN,gy8y,Negative,Undetermined
62454,M11,N gene,FAM,2.165470,32.24727452491577,12500.000000,3,22,45,169153.468750,67533.791947,169153.470,AC00GY8Y,71346,S1075521,SARS00032181,Clinical Sample,Positive,Inconclusive,1/21/22,Submitted Sample,SARS00032181- Retest 1,NaN,gy8y,Positive,32.24727452
64929,O19,RnaseP,ATTO 647,0.000000,Undetermined,10000.000000,3,44,45,47670.582031,-83.409830,47670.582,AC00GY8Y,71786,S1075537,SARS00035532,Clinical Sample,Positive,Inconclusive,1/21/22,Submitted Sample,SARS00035532- Retest 1,NaN,gy8y,Negative,Undetermined
64974,O19,E gene,VIC,1.624481,Undetermined,6000.000000,3,44,45,458569.312500,907.977751,458569.300,AC00GY8Y,71018,S1075537,SARS00035532,Clinical Sample,Positive,Inconclusive,1/21/22,Submitted Sample,SARS00035532- Retest 1,NaN,gy8y,Negative,Undetermined
65019,O19,N gene,FAM,2.033873,32.13139229330451,12500.000000,3,21,45,182505.062500,29391.834128,182505.060,AC00GY8Y,71402,S1075537,SARS00035532,Clinical Sample,Positive,Inconclusive,1/21/22,Submitted Sample,SARS00035532- Retest 1,NaN,gy8y,Positive,32.13139229
137154,M11,RnaseP,ATTO 647,2.486534,25.346094898141985,10000.000000,3,18,45,198663.250000,172013.921467,198663.250,AC00GY79,68274,S785146,SARS00005334,Clinical Sample,Positive,Inconclusive,9/13/2021,Submitted Sample,SARS00005334- Retest 1,NaN,gy79,Positive,25.346094898141985


In [87]:
join_df.pcr_plate.nunique()

115

In [107]:

df = join_df[(join_df.sample_type == 'Buffer Negative Control (Extraction)') & (join_df.cycle_no == 1)]
df['igi_call_encoded'] = [1 if x == 'Positive' else 0 for x in df['igi_call']]
df1 = df.groupby('sample_id').igi_call_encoded.sum().reset_index()
df1[df1.igi_call_encoded > 0].shape

/var/folders/q4/trxghvfn21sdx12kbtpt92n00000gn/T/ipykernel_57387/3698874662.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['igi_call_encoded'] = [1 if x == 'Positive' else 0 for x in df['igi_call']]


(61, 2)

In [120]:
control_gene_ids = (join_df
                    .loc[join_df.retest_sample_id_1.isna()]
 .loc[join_df.cycle_no == 1]
 .loc[join_df.target.isin(['RnaseP', 'MS2'])]).curve_idx.unique()

other_ground_truth = (join_df
                      .loc[join_df.retest_sample_id_1.isna()]
 .loc[~join_df.target.isin(['RnaseP','MS2'])]
 .loc[join_df.sample_type != 'Clinical Sample']).curve_idx.unique()


In [121]:
len(np.concatenate([control_gene_ids, other_ground_truth]))

23671

In [122]:
(join_df[join_df.curve_idx.isin(list(np.concatenate([control_gene_ids, other_ground_truth])))]
 .groupby(['sample_type','igi_call','target'])
 .curve_idx
 .nunique())

sample_type                                 igi_call  target
Buffer Negative Control (Extraction)        Negative  E gene      169
                                                      MS2           6
                                                      N gene      232
                                                      ORF1ab       64
                                                      RnaseP      166
                                                      S gene       64
                                            Positive  MS2          58
                                                      N gene        1
                                                      RnaseP        3
Clinical Sample                             Negative  MS2         134
                                                      RnaseP      403
                                            Positive  MS2        4593
                                                      RnaseP    14004
Human Normal Negative Control

In [127]:
control_gene_ids = (join_df
                    .loc[(~join_df.retest_sample_id_1.isna()) & (join_df.retest_sample_id_2.isna())]
 .loc[join_df.cycle_no == 1]
 .loc[join_df.target.isin(['RnaseP', 'MS2'])]).curve_idx.unique()

other_ground_truth = (join_df
                      .loc[(~join_df.retest_sample_id_1.isna()) & (join_df.retest_sample_id_2.isna())]
 .loc[~join_df.target.isin(['RnaseP','MS2'])]
 .loc[join_df.sample_type != 'Clinical Sample']).curve_idx.unique()

(join_df[join_df.curve_idx.isin(list(np.concatenate([control_gene_ids, other_ground_truth])))]
 .groupby(['sample_type','final_patient_result','current_sample_result'])
 .sample_id
 .nunique())


sample_type      final_patient_result  current_sample_result
Clinical Sample  Negative              Inconclusive             235
                                       Invalid                  549
                                       Negative                   6
                 Positive              Inconclusive              56
                                       Invalid                   47
                                       Positive                   1
                 ReSample              Inconclusive               9
                                       Invalid                   13
Name: sample_id, dtype: int64

In [128]:
control_gene_ids = (join_df
                    .loc[(~join_df.retest_sample_id_1.isna()) & (~join_df.retest_sample_id_2.isna())]
 .loc[join_df.cycle_no == 1]
 .loc[join_df.target.isin(['RnaseP', 'MS2'])]).curve_idx.unique()

other_ground_truth = (join_df
                      .loc[(~join_df.retest_sample_id_1.isna()) & (~join_df.retest_sample_id_2.isna())]
 .loc[~join_df.target.isin(['RnaseP','MS2'])]
 .loc[join_df.sample_type != 'Clinical Sample']).curve_idx.unique()

(join_df[join_df.curve_idx.isin(list(np.concatenate([control_gene_ids, other_ground_truth])))]
 .groupby(['sample_type','final_patient_result','current_sample_result'])
 .sample_id
 .nunique())


sample_type      final_patient_result  current_sample_result
Clinical Sample  Negative              Inconclusive             21
                                       Invalid                  74
                                       Negative                  1
                 Positive              Inconclusive              2
                                       Invalid                   3
                                       Negative                  1
                 ReSample              Inconclusive              1
                                       Invalid                  14
Name: sample_id, dtype: int64